In [16]:
import os 
from dotenv import load_dotenv, find_dotenv
_=load_dotenv(find_dotenv())
groq_api_key = os.environ["GROQ_API_KEY"]

In [17]:
from langchain_groq import ChatGroq

model = ChatGroq(model="openai/gpt-oss-safeguard-20b")

In [18]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

In [19]:
prompt = ChatPromptTemplate.from_template("tell me a curious fact about {soccer_player}")

output_parser = StrOutputParser()

In [20]:
chain = prompt | model | output_parser

chain.invoke({"soccer_player":"Ronaldo"})

'**Curious fact about Cristiano Ronaldo**\n\nHe’s the only player ever to win *all four* of Europe’s top‑tier club trophies: the Premier\u202fLeague (with Manchester\u202fUnited), La\u202fLiga (with Real\u202fMadrid), Serie\u202fA (with Juventus), and the UEFA Champions\u202fLeague (with both Manchester\u202fUnited and Real\u202fMadrid). In other words, he’s the sole footballer who’s lifted the top domestic league in England, Spain and Italy *and* the continent’s most prestigious club competition.'

### Use of .bind() to add arguments to a Runnable in a LCEL Chain

In [21]:
chain = prompt | model.bind(stop=["is"]) | output_parser
chain.invoke({"soccer_player":"Ronaldo"})

''

# Combining LCEL Chains

In [22]:
from langchain_groq import ChatGroq

model = ChatGroq(model = "openai/gpt-oss-safeguard-20b")

In [23]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("tell me a sentence about {politician}")

model = ChatGroq(model="openai/gpt-oss-safeguard-20b")

chain = prompt | model | StrOutputParser()

In [24]:
chain.invoke("Chamberlain")

'Neville\u202fChamberlain, who served as Prime Minister of the United Kingdom from 1937 to 1940, is most famously remembered for signing the Munich Agreement in 1938, a controversial act that sought to appease Adolf\u202fHitler.'

### Combined Chain

In [25]:
historian_prompt = ChatPromptTemplate.from_template("Was {politician} positive for Humanity?")

composed_chain = {"politician":chain} | historian_prompt | model | StrOutputParser()

In [26]:
composed_chain.invoke({"politician":"Lincoln"})

'Abraham\u202fLincoln’s legacy is overwhelmingly viewed as a positive one for humanity, primarily because of two landmark achievements:\n\n| Action | Human‑rights impact | Broader significance |\n|--------|---------------------|----------------------|\n| **Emancipation Proclamation (1863)** | Declared all enslaved people in Confederate territory “forever free.” It shifted the war’s purpose to a moral crusade against slavery and paved the way for the 13th Amendment, which abolished slavery nationwide. | Set a precedent that a national government can use executive power to advance human rights. |\n| **Preserving the Union** | By defeating the Confederacy, Lincoln kept the United States intact, preventing the fragmentation that could have led to further regional conflicts and a less stable world order. | Maintained a large, unified nation that could later become a global advocate for civil rights and democratic norms. |\n\n### Why it’s considered positive\n\n1. **End of slavery** – Lincol

### Another example: a chain inside a chain

In [31]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from operator import itemgetter

prompt1 = ChatPromptTemplate.from_template("Which country do the {politician} come from?")
prompt2 = ChatPromptTemplate.from_template("What continent is the country {country} in? respond in {language}")

model = ChatGroq(model="openai/gpt-oss-safeguard-20b")

chain1 = prompt1 | model | StrOutputParser()

chain2 = (
    {
        "country": chain1,
        "language": itemgetter("language"),
    }
    | prompt2
    | model
    | StrOutputParser()
)

chain2.invoke({"politician":"Miterrand" , "language": "Nepali"})

'फ्रान्स यूरोप महादेशमा अवस्थित छ।'

In [32]:
from langchain_groq import ChatGroq

model = ChatGroq(model = "openai/gpt-oss-safeguard-20b")

In [ ]:
import bs4
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)

docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

splits = text_splitter.split_documents(docs)

vectorstore = Chroma.from_documents(documents=splits, embedding=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2"))

retriever = vectorstore.as_retriever()

prompt = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks.
Use the following context to answer the question.
If you do not know the answer, say that you do not know.
Keep the answer concise.

Context:
{context}

Question:
{question}

Answer:"""
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [35]:
rag_chain.invoke("What is Task Decomposition?")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


'Task decomposition is the process of breaking a complex task into a sequence of smaller, more manageable sub‑tasks or steps (subgoals), enabling an agent to plan and execute the overall goal more effectively.'